# Potrzebne biblioteki do zainstalowania

In [1]:
%pip install \
numpy \
cvxpy \
ecos \
scs \
galois \
matplotlib \
scikit-learn \
pyqt5

  Using cached ecos-2.0.14.tar.gz (142 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached galois-0.4.10-py3-none-any.whl.metadata (14 kB)
  Using cached PyQt5-5.15.11-cp38-abi3-macosx_11_0_arm64.whl.metadata (2.1 kB)
  Using cached clarabel-0.11.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (4.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached pyqt5_qt5-5.15.18-py3-none-macosx_11_0_arm64.whl.metadata (536 bytes)
  Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
  Using cached markupsafe-3.0

### Importujemy biblioteki

In [2]:
from __future__ import annotations
from dataclasses import dataclass
from itertools import combinations
from typing import List, Tuple, Dict
import numpy as np
import cvxpy as cp
import galois
import random

import matplotlib
matplotlib.use("QtAgg")

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.patches import Polygon
from sklearn.decomposition import PCA

### Struktura sympleksu

In [3]:
@dataclass(frozen=True)
class Simplex:
    """
    Reprezentacja sympleksu:
      - dim: wymiar (0 -> wierzchołek, 1 -> krawędź, 2 -> trójkąt, ...)
      - vertices: indeksy wierzchołków (długość = dim + 1)
    """
    dim: int
    vertices: Tuple[int, ...]

### Funkcja do generowania losowych punktów

In [4]:
def generate_points_uniform(
    n_points: int,
    ambient_dim: int,
    low: float = 0.0,
    high: float = 1.0,
    seed: int = 0
) -> np.ndarray:
    """
    Generuje n_points punktów w R^ambient_dim,
    równomiernie w hiper-sześcianie [low, high]^d.
    """
    if seed==0:
        seed=random.randint(1,100)
    rng = np.random.default_rng(seed)
    return rng.uniform(low, high, size=(n_points, ambient_dim))

### Funkcja sprawdzająca przecinanie się kul

In [5]:
def balls_have_common_intersection(P: np.ndarray, r: float, solver: str = "ECOS") -> bool:
    """
    Sprawdza, czy przecięcie kul (zamkniętych) o promieniu r i środkach w P jest niepuste

    Robi to przez problem wykonalności:
        znajdź x takie, że ||x - P[i]|| <= r dla wszystkich i.
    Jeśli problem jest wykonalny -> True, w przeciwnym razie -> False.
    """
    P = np.asarray(P, dtype=float)
    m, d = P.shape

    # Dla 0 lub 1 punktu przecięcie zawsze istnieje.
    if m <= 1:
        return True

    x = cp.Variable(d)
    constraints = [cp.norm(x - P[i], 2) <= r for i in range(m)]
    prob = cp.Problem(cp.Minimize(0), constraints)

    # ECOS jest szybki dla takich problemów; SCS jako zapas.
    try:
        prob.solve(solver=solver, warm_start=True)
    except Exception:
        prob.solve(solver="SCS", warm_start=True)

    return prob.status in ("optimal", "optimal_inaccurate")

In [6]:
@dataclass(frozen=True)
class FiltrationEvent:
    """
    - radius: "prawdziwy" promień (geometria Čech)
    - time:   iniekcyjny czas jako liczba naturalna (0,1,2,...)
    - simplex: sympleks
    """
    radius: float
    time: int
    simplex: Simplex
	
def simplex_filtration_value(P: np.ndarray, solver: str = "ECOS") -> float:
    """
    Liczy minimalny promień f(sigma), przy którym kule o środkach P zaczynają mieć część wspólną.

    Matematycznie:
        f = min_x max_i ||x - P[i]||

    SOCP:
        min t
        s.t. ||x - P[i]|| <= t  dla wszystkich i
        t >= 0
    """
    P = np.asarray(P, dtype=float)
    m, d = P.shape

    if m <= 1:
        return 0.0
    if m == 2:
        return float(np.linalg.norm(P[0] - P[1]) / 2.0)

    x = cp.Variable(d)
    t = cp.Variable(nonneg=True)
    constraints = [cp.norm(x - P[i], 2) <= t for i in range(m)]
    prob = cp.Problem(cp.Minimize(t), constraints)

    try:
        prob.solve(solver=solver, warm_start=True)
    except Exception:
        prob.solve(solver="SCS", warm_start=True)

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"SOCP dla filtracji nie powiódł się, status={prob.status}")

    return float(t.value)


In [7]:
class CechComplex:
    """
    Minimalny "silnik" do Twojego projektu:
      - trzyma punkty w R^n
      - buduje kompleks Čech dla jednego wspólnego promienia r
      - przechowuje sympleksy jako (dim, vertices)
      - ma funkcję do wyliczenia f(sigma)
    """

    def __init__(self, points: np.ndarray):
        self.points = np.asarray(points, dtype=float)
        if self.points.ndim != 2:
            raise ValueError("points musi mieć kształt (N, d)")
        self.N, self.d = self.points.shape

        self.simplices: List[Simplex] = []
        # cache na f(vertices) -> czas (żeby nie liczyć 100x tego samego)
        self._filtration_cache: Dict[Tuple[int, ...], float] = {}

    def build_for_radius(self, r: float, max_dim: int = 2, solver: str = "ECOS") -> List[Simplex]:
        """
        Buduje kompleks Čech(A, r) do wymiaru max_dim (włącznie),
        sprawdzając przecięcia kul promienia r.
        """
        r = float(r)
        max_dim = int(max_dim)

        self._pairwise = np.linalg.norm(self.points[:, None, :] - self.points[None, :, :], axis=2)


        simplices: List[Simplex] = []

        # 0-sympleksy
        for i in range(self.N):
            simplices.append(Simplex(dim=0, vertices=(i,)))

        # wyższe wymiary
        for dim in range(1, max_dim + 1):
            for verts in combinations(range(self.N), dim + 1):
                P = self.points[list(verts)]
                # szybki warunek konieczny: wszystkie pary <= 2r
                Dmax = self._pairwise[np.ix_(verts, verts)].max()
                if Dmax > 2*r:
                    continue

                if balls_have_common_intersection(P, r, solver=solver):
                    simplices.append(Simplex(dim=dim, vertices=verts))

        self.simplices = simplices
        print(f"Zbudowano Čech dla R={r}, N={self.N}, d={self.d}, n={max_dim}")
        print(f"Liczba sympleksów: {len(self.simplices)}")
        return simplices

    def filtration_value(self, simplex: Simplex, solver: str = "ECOS") -> float:
        """
        Zwraca minimalny promień f(simplex) (czas pojawienia w filtracji Čech).
        Z cachem.
        """
        if simplex.dim == 0:
            return 0.0

        key = simplex.vertices
        if key in self._filtration_cache:
            return self._filtration_cache[key]

        P = self.points[list(key)]
        t = float(simplex_filtration_value(P, solver=solver))
        self._filtration_cache[key] = t
        return t

    def events_from_simplices(self, simplices: List[Simplex], solver: str = "ECOS") -> List[FiltrationEvent]:
        """
        1) Liczy promień pojawienia radius dla podanych sympleksów
        2) Wymusza monotoniczność po ścianach: radius(ściana) <= radius(sympleks)
        3) Robi filtrację jako INIEKCJĘ do N: time = 0,1,2,... wg porządku:
             radius -> dim -> vertices
        """
        # (A) surowe promienie
        raw_radius: Dict[Tuple[int, ...], float] = {}
        for s in simplices:
            raw_radius[s.vertices] = float(self.filtration_value(s, solver=solver))

        # (B) monotoniczność po ścianach
        simplices_sorted = sorted(simplices, key=lambda s: (s.dim, s.vertices))
        radius = dict(raw_radius)

        for s in simplices_sorted:
            if s.dim == 0:
                radius[s.vertices] = 0.0
                continue

            max_face = 0.0
            for face in combinations(s.vertices, s.dim):  # wszystkie (dim-1)-ściany
                if face in radius:
                    max_face = max(max_face, radius[face])

            if radius[s.vertices] < max_face:
                radius[s.vertices] = max_face

        # (C) totalny porządek
        # round tylko po to, żeby "prawie równe" radii (błędy solvera) traktować jako równe
        EPS_DIGITS = 10
        ordered = sorted(
            simplices,
            key=lambda s: (round(radius[s.vertices], EPS_DIGITS), s.dim, s.vertices)
        )

        # (D) time = liczby naturalne
        events: List[FiltrationEvent] = []
        for t, s in enumerate(ordered):
            events.append(FiltrationEvent(radius=float(radius[s.vertices]), time=t, simplex=s))
   
        return events
def simplices_from_events(events: List[FiltrationEvent], r: float):
    r = float(r)
    return [e.simplex for e in events if e.radius <= r]
def print_filter(events):
    for e in events:
        print(f"K = {e.simplex.vertices} --> {e.time:.1f},  (R = {e.radius:.5f})")


### Funkcje do wizualizacji kompleksów n=2

In [8]:
def _triangle_min_enclosing_radius_2d(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    Minimalny promień kuli w R^2, która zawiera 3 punkty (a,b,c).
    Dla 3 punktów to:
      - jeśli trójkąt jest rozwarty/prosty -> połowa najdłuższego boku
      - jeśli ostrokątny -> promień okręgu opisanego: R = abc/(4A)
    """
    ab = np.linalg.norm(a - b)
    bc = np.linalg.norm(b - c)
    ca = np.linalg.norm(c - a)

    # sortujemy boki: s0 <= s1 <= s2
    s = np.sort(np.array([ab, bc, ca], dtype=float))
    s0, s1, s2 = s

    # degeneracja / prawie współliniowe
    # pole = |cross(b-a, c-a)|/2
    cross = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    area2 = abs(cross)  # to jest 2*A
    if area2 < 1e-15:
        return 0.5 * s2  # w linii: min okrąg daje średnicę na najdłuższym odcinku

    # test na rozwartość/prostokątność: s0^2 + s1^2 <= s2^2
    if s0 * s0 + s1 * s1 <= s2 * s2 + 1e-15:
        return 0.5 * s2

    # ostrokątny: R = abc/(4A) = abc/(2*area2)
    R = (ab * bc * ca) / (2.0 * area2)
    return float(R)


def precompute_edges_triangles_2d(points: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Precomputuje wszystkie krawędzie i trójkąty dla punktów w R^2.

    Zwraca:
      edges:     array (M, 3)  gdzie [i, j, r_ij]
      triangles: array (T, 4)  gdzie [i, j, k, r_ijk]
    """
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("precompute_edges_triangles_2d działa tylko dla (N,2)")

    N = points.shape[0]

    # --- edges
    edge_list = []
    for i in range(N):
        pi = points[i]
        for j in range(i + 1, N):
            r = 0.5 * np.linalg.norm(pi - points[j])
            edge_list.append((i, j, r))
    edges = np.array(edge_list, dtype=float)  # (M,3)

    # --- triangles
    tri_list = []
    for i in range(N):
        a = points[i]
        for j in range(i + 1, N):
            b = points[j]
            for k in range(j + 1, N):
                c = points[k]
                r = _triangle_min_enclosing_radius_2d(a, b, c)
                tri_list.append((i, j, k, r))
    triangles = np.array(tri_list, dtype=float)  # (T,4)

    return edges, triangles


def plot_cech_2d_two_sliders_fast(points: np.ndarray, R_max: float, title: str = "Čech w R^2 (fast)") -> None:
    """
    Dwa suwaki:
      - r (0..R_max)
      - k (1..N)

    Szybkość: precompute wszystkich krawędzi i trójkątów raz,
    potem tylko maski NumPy.
    """
    points = np.asarray(points, dtype=float)
    N = points.shape[0]
    R_max = float(R_max)

    # precompute raz
    edges, triangles = precompute_edges_triangles_2d(points)

    # indeksy i promienie jako osobne tablice
    e_i = edges[:, 0].astype(int)
    e_j = edges[:, 1].astype(int)
    e_r = edges[:, 2]

    t_i = triangles[:, 0].astype(int)
    t_j = triangles[:, 1].astype(int)
    t_k = triangles[:, 2].astype(int)
    t_r = triangles[:, 3]

    # figura
    fig, ax = plt.subplots()
    plt.subplots_adjust(bottom=0.25)

    scat = ax.scatter(points[:, 0], points[:, 1], zorder=4)
    texts = []
    for idx, (x, y) in enumerate(points):
        txt = ax.text(x, y, str(idx), fontsize=9, zorder=5)
        texts.append(txt)

    edge_collection = LineCollection([], linewidths=1.8, zorder=3)
    tri_collection = PolyCollection([], alpha=0.25, linewidths=0.8, zorder=2)
    ax.add_collection(edge_collection)
    ax.add_collection(tri_collection)

    # osie
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    half = span / 2.0 if span > 0 else 1.0
    ax.set_xlim(center[0] - half, center[0] + half)
    ax.set_ylim(center[1] - half, center[1] + half)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    # suwaki
    k0 = min(N, 10)
    r0 = 0.0

    ax_r = fig.add_axes([0.15, 0.12, 0.70, 0.04])
    slider_r = Slider(ax=ax_r, label="r", valmin=0.0, valmax=R_max, valinit=r0)

    ax_k = fig.add_axes([0.15, 0.06, 0.70, 0.04])
    slider_k = Slider(ax=ax_k, label="k (liczba wierzchołków)", valmin=1, valmax=N, valinit=k0, valstep=1)

    def redraw():
        r = float(slider_r.val)
        k = int(slider_k.val)

        # pokaż/ukryj podpisy
        for idx, txt in enumerate(texts):
            txt.set_visible(idx < k)

        # --- edges mask: i<k, j<k, r_ij<=r
        mask_e = (e_i < k) & (e_j < k) & (e_r <= r)
        sel_e_i = e_i[mask_e]
        sel_e_j = e_j[mask_e]

        segs = [[points[i], points[j]] for i, j in zip(sel_e_i, sel_e_j)]
        edge_collection.set_segments(segs)

        # --- triangles mask
        mask_t = (t_i < k) & (t_j < k) & (t_k < k) & (t_r <= r)
        sel_t_i = t_i[mask_t]
        sel_t_j = t_j[mask_t]
        sel_t_k = t_k[mask_t]

        polys = [[points[i], points[j], points[m]] for i, j, m in zip(sel_t_i, sel_t_j, sel_t_k)]
        tri_collection.set_verts(polys)

        # показываем только активные точки
        scat.set_offsets(points[:k])

        ax.set_title(f"{title} | k={k} | r={r:.4f} | edges={mask_e.sum()} | tris={mask_t.sum()}")
        fig.canvas.draw_idle()

    slider_r.on_changed(lambda _: redraw())
    slider_k.on_changed(lambda _: redraw())

    redraw()
    plt.show()

### Funkcja do wizualizacji kompleksów n>2

In [21]:
def draw_complex3(simplices,points):
    # projekcja na 2 wymiar
    pca = PCA(n_components=2)
    points_2d=pca.fit_transform(points)
    fig, ax = plt.subplots(figsize=(10, 10))

    #rysujemy punkty (wierzcholki)
    ax.scatter(points_2d[:, 0],points_2d[:, 1],s=30,color="black",zorder=4)

    #sortujemy kompleks po wymirach dla pewnosci
    simplices = sorted(simplices,key=lambda s: s.dim)
    faces=[]
    for simplex in simplices:
        verts=simplex.vertices
        c=points_2d[list(verts)]    #bierzemy koordynaty danego sympleksu
        if simplex.dim==0:  #juz narysowane
            continue
        elif simplex.dim==1:    #krawedzie
            ax.plot(c[:, 0],c[:, 1],color="black",linewidth=1.5,alpha=0.8,zorder=3)
        elif simplex.dim==2:    #trojkaty
            poly = Polygon(c,closed=True,facecolor="blue",alpha=0.4,zorder=1)
            ax.add_patch(poly)
        else:
            # im wyzszy wymiar tym troche ciemniejszy kolor niebieskiego, malujemy cale pole pomiedzy wierzcholkami
            a=min(1,0,5+simplex.dim*0.05)
            for triangle in combinations(verts, 3):
                t_c=points_2d[list(triangle)]
                poly=Polygon(t_c,closed=True,facecolor="blue",edgecolor=None,alpha=a,zorder=2)
                ax.add_patch(poly)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Kompleks symplicjalny")
    plt.show()

In [10]:
def plot_for_dimension(points: np.ndarray,simplices: List[Simplex],events: List[FiltrationEvent] | None = None,R: float | None = None,max_dim: int = 2) -> None:
    """
    Funkcja TYLKO do rysowania.

    - jeśli d==2: wymaga `events` (filtracja z suwakiem)
    - jeśli d!=2: rysuje `simplices` (rzut PCA do 2D)
    """
    points = np.asarray(points, dtype=float)
    d = points.shape[1]

    if d == 2:
       plot_cech_2d_two_sliders_fast(points, R_max=R, title="Čech w R^2 (r + k)")
    else:
        draw_complex3(simplices, points)

### Funkcja do obliczania homologii symplicjlanej nad $Z_2$

In [11]:
def simplical_homology(complex,n):
    complex=sorted(complex, key=lambda x: x.dim) #sortujemy sympleksy dla pewnosci zeby byly w kolejnosci od najmniejszych wymiarowo
    dimension=complex[-1].dim               #najwyzszy wymiar sympelksu w kompleksie
    dimCn=np.zeros(dimension+1,dtype=int)   #tablica ilosci n-sympleksow
    dimRn=np.zeros(dimension+2,dtype=int)   #tablica rang macierzy operatorow brzegu

    for i in range(len(complex)):   #zliczamy ilosc n-sympleksow
        dimCn[complex[i].dim]+=1
    iter_n=dimCn[0]     #zmienne potrzebne dla poprawnego indeksowania listy sympleksow
    iter_n_1=0
    for i in range(1,dimension+1):
        operator_matrix=np.zeros((dimCn[i-1],dimCn[i]),dtype=int)   #tworzymy macierz operatoru brzegu o odpowiednich wymiarach z samych 0
        for j in range(dimCn[i]):
            simplices=complex[j+iter_n].vertices    #bierzemy sympleks, bedziemy sprawdzac ktore sympleksy sie w nim zawieraja z mniejszego wymiaru
            for t in range(dimCn[i-1]):
                if(set(complex[t+iter_n_1].vertices)<=set(simplices)):  #sympleks o mniejszym wymiarze zawiera sie w sympleksie, zmieniamy w amcierzy odpowiednio na 1
                    operator_matrix[t,j]=1
        iter_n+=dimCn[i]
        iter_n_1+=dimCn[i-1]
        Z2=galois.GF(2)     #tworzymy przestrzen modulo 2, zeby poprawnie range macierzy
        operator_matrixZ2=Z2(operator_matrix)
        dimRn[i]=np.linalg.matrix_rank(operator_matrixZ2)

    for i in range(n+1):
        if i>dimension:
            print(f"H_{i}=0")
        else:        #obliczamy liczbe bettiego i odpowiednio wypisujemy
            betty=dimCn[i]-dimRn[i]-dimRn[i+1]
            if betty==0:
                print(f"H_{i}=0")
            else:
                print(f"H_{i}=Z2^{int(betty)}")

### Krzywa charakterystyki Eulera

In [12]:
def plot_euler_characteristic_curve(cech: CechComplex, R_max: float, max_dim: int = 2):
    """
    Rysuje krzywą charakterystyki Eulera:
        ECC(t) = χ(K(t))

    gdzie K(t) to podkompleks Čecha otrzymany w filtracji f.

    Parametry:
    ----------
    cech : CechComplex
        Kompleks Čecha zbudowany na chmurze punktów
    R_max : float
        Maksymalny promień filtracji
    max_dim : int
        Maksymalny wymiar sympleksów (domyślnie 2)
    """

    # pobieramy zdarzenia filtracji
    events = cech.filtration_events_up_to(R_max=R_max, max_dim=max_dim)

    # sortowanie po czasie (na wszelki wypadek)
    events.sort(key=lambda e: e.time)

    times = []
    euler_values = []

    chi = 0  # aktualna charakterystyka Eulera

    for e in events:
        dim = e.simplex.dim
        chi += (-1) ** dim

        times.append(e.time)
        euler_values.append(chi)

    # rysowanie jako funkcja schodkowa
    plt.figure()
    plt.step(times, euler_values, where="post")
    plt.xlabel("t (parametr filtracji r)")
    plt.ylabel("χ(K(t))")
    plt.title("Krzywa charakterystyki Eulera (ECC)")
    plt.grid(True)
    plt.show()

In [40]:
def plot_euler_characteristic_curve_time(events, title: str = "ECC(time)") -> None:
    """
    Rysuje ECC jako funkcję czasu filtracji time=0,1,2,...
    """
    ev = sorted(events, key=lambda e: e.time)

    t_vals = [-1]
    chi_vals = [0]
    chi = 0

    for e in ev:
        chi += (-1) ** e.simplex.dim
        t_vals.append(e.time)
        chi_vals.append(chi)

    plt.figure()
    plt.step(t_vals, chi_vals, where="post")
    plt.xlabel("f (0,1,2,...)")
    plt.ylabel("χ(K_f)")
    # plt.title()
    plt.grid(True)
    plt.show()

In [55]:
def plot_euler_for_complex(complex):
    complex=sorted(complex, key=lambda x: x.dim)

    t_vals = [-1]
    chi_vals = [0]
    chi = 0

    for (i,simplices) in enumerate(complex):
        chi += (-1) ** simplices.dim
        t_vals.append(i)
        chi_vals.append(chi)

    plt.figure()
    plt.step(t_vals, chi_vals, where="post")
    plt.xlabel("f (0,1,2,...)")
    plt.ylabel("χ(K_f)")
    plt.grid(True)
    plt.show()


## Przypadki testowe dla liczenia homologii symplicjlanych

### Sfera

In [22]:
def test_sphere():
    """
    Test homologii dla sfery S^2.
    Realizacja: powierzchnia czworościanu
    Homologie nad Z2:
        H_0 = Z2
        H_1 = 0
        H_2 = Z2
    """

    # 0-sympleksy
    v0 = Simplex(dim=0, vertices=(0,))
    v1 = Simplex(dim=0, vertices=(1,))
    v2 = Simplex(dim=0, vertices=(2,))
    v3 = Simplex(dim=0, vertices=(3,))

    # 1-sympleksy 
    e01 = Simplex(dim=1, vertices=(0,1))
    e02 = Simplex(dim=1, vertices=(0,2))
    e03 = Simplex(dim=1, vertices=(0,3))
    e12 = Simplex(dim=1, vertices=(1,2))
    e13 = Simplex(dim=1, vertices=(1,3))
    e23 = Simplex(dim=1, vertices=(2,3))

    # 2-sympleksy 
    t012 = Simplex(dim=2, vertices=(0,1,2))
    t013 = Simplex(dim=2, vertices=(0,1,3))
    t023 = Simplex(dim=2, vertices=(0,2,3))
    t123 = Simplex(dim=2, vertices=(1,2,3))

    # kompleks symplicjalny sfery
    sphere_complex = [
        v0, v1, v2, v3,
        e01, e02, e03, e12, e13, e23,
        t012, t013, t023, t123
    ]

    print("\n=== Test: sfera S^2 ===")
    simplical_homology(sphere_complex, n=2)
test_sphere()


=== Test: sfera S^2 ===
H_0=Z2^1
H_1=0
H_2=Z2^1


### Torus

In [23]:
def test_torus():
    """
    Triangulacja torusa siatka 3x3 kwadraty, każdy kwadrat dzielony na dwa trójkąty.
    Homologie Z2:
        H_0 = Z2
        H_1 = Z2^2
        H_2 = Z2
    """
    # wierzcholki
    v = [Simplex(0, (i,)) for i in range(9)]

    # zawijanie indeksow
    def idx(i,j):
        return (i % 3) + 3*(j % 3)

    edges_set = set()

    #  trójkąty (2 na każdy kwadrat)
    triangles = []
    for j in range(3):
        for i in range(3):
            # wierzchołki kwadratu
            a = idx(i,j)
            b = idx(i+1,j)
            c = idx(i,j+1)
            d = idx(i+1,j+1)
            # dwa trójkąty dzielące kwadrat
            triangles.append(Simplex(2, (a,b,d)))
            triangles.append(Simplex(2, (a,d,c)))
            # dodajemy krawędzie do zbioru
            edges_set.update([
                tuple(sorted((a,b))),
                tuple(sorted((b,d))),
                tuple(sorted((d,a))),
                tuple(sorted((a,c))),
                tuple(sorted((c,d))),
            ])

    # krawędzie
    edges = [Simplex(1, e) for e in edges_set]

    torus_complex = v + edges + triangles

    print("\n=== Test: Torus 3x3 kwadratów ===")
    simplical_homology(torus_complex, n=2)
test_torus()


=== Test: Torus 3x3 kwadratów ===
H_0=Z2^1
H_1=Z2^2
H_2=Z2^1


#### Przestrzeń rzutowa

In [24]:
def test_RP2():
    """
    Triangulacja przestrzeni rzutowej RP^2.
    Homologie Z2:
        H0 = Z2
        H1 = Z2
        H2 = Z2
    """

    # wierzcholki
    v = [Simplex(0, (i,)) for i in range(6)]

    # krawędzie
    edges_list = [
        (0,1),(0,2),(0,3),(0,4),(0,5),
        (1,2),(1,3),(1,5),
        (2,3),(2,4),
        (3,4),(3,5),
        (4,5),
        (1,4),(2,5)
    ]
    edges = [Simplex(1, e) for e in edges_list]

    # trójkąty
    triangles_list = [
        (0,1,2),(0,2,3),(0,3,4),(0,4,5),(0,5,1),
        (1,2,5),(2,3,5),(3,4,5),(1,3,4),(1,4,5)
    ]
    triangles = [Simplex(2, t) for t in triangles_list]

    rp2_complex = v + edges + triangles

    print("\n=== Test: RP^2  ===")
    simplical_homology(rp2_complex, n=2)
test_RP2()


=== Test: RP^2  ===
H_0=Z2^1
H_1=Z2^1
H_2=Z2^1


## Przypadki testowe dla krzywej Eulera

#### Krzywa eulera dla sfery

In [57]:
def euler_curve_sphere():
    v_sphere = [Simplex(0,(i,)) for i in range(4)]
    edges_sphere = [Simplex(1, (0,1)), Simplex(1,(0,2)), Simplex(1,(0,3)),
                    Simplex(1,(1,2)), Simplex(1,(1,3)), Simplex(1,(2,3))]
    triangles_sphere = [Simplex(2,(0,1,2)), Simplex(2,(0,1,3)), Simplex(2,(0,2,3)), Simplex(2,(1,2,3))]
    sphere_complex = v_sphere + edges_sphere + triangles_sphere

    plot_euler_for_complex(sphere_complex)
euler_curve_sphere()

#### Krzywa eulera dla torusa

In [58]:
def euler_curve_torus():
    v_torus = [Simplex(0, (i,)) for i in range(9)]
    def idx(i,j): return (i % 3) + 3*(j % 3)
    edges_set = set()
    triangles_torus = []
    for j in range(3):
        for i in range(3):
            a = idx(i,j); b = idx(i+1,j); c = idx(i,j+1); d = idx(i+1,j+1)
            triangles_torus.append(Simplex(2,(a,b,d)))
            triangles_torus.append(Simplex(2,(a,d,c)))
            edges_set.update([tuple(sorted((a,b))), tuple(sorted((b,d))), tuple(sorted((d,a))),
                              tuple(sorted((a,c))), tuple(sorted((c,d)))])
    edges_torus = [Simplex(1,e) for e in edges_set]
    torus_complex = v_torus + edges_torus + triangles_torus
    plot_euler_for_complex(torus_complex)
    
euler_curve_torus()

#### Krzywa eulera dla przestrzeni rzutowej

In [59]:
def euler_curve_rp2():
    v_rp2 = [Simplex(0, (i,)) for i in range(6)]
    edges_list = [
        (0,1),(0,2),(0,3),(0,4),(0,5),
        (1,2),(1,3),(1,5),
        (2,3),(2,4),
        (3,4),(3,5),
        (4,5),
        (1,4),(2,5)
    ]
    edges_rp2 = [Simplex(1,e) for e in edges_list]
    triangles_list = [
        (0,1,2),(0,2,3),(0,3,4),(0,4,5),(0,5,1),
        (1,2,5),(2,3,5),(3,4,5),(1,3,4),(1,4,5)
    ]
    triangles_rp2 = [Simplex(2,t) for t in triangles_list]
    rp2_complex = v_rp2 + edges_rp2 + triangles_rp2
    plot_euler_for_complex(rp2_complex)
euler_curve_rp2()

## Zróbmy przykład dla losowego zbioru

In [64]:
n=3 # wymiar
k=14 # liczba punktów
r=0.2 # promień

#losujemy z przedizału (0,1)
points=generate_points_uniform(n_points=k,low=0,high=1,ambient_dim=n,seed=5)

cech_complex=CechComplex(points)
simplices=cech_complex.build_for_radius(r=r,max_dim=n)
simplical_homology(simplices,n)

Zbudowano Čech dla R=0.2, N=14, d=3, n=3
Liczba sympleksów: 33
H_0=Z2^3
H_1=Z2^2
H_2=0
H_3=0


In [63]:
plot_for_dimension(points=points,simplices=simplices,R=r,max_dim=n)

In [ ]:
events=cech_complex.events_from_simplices(simplices)
print_filter(events)

In [66]:
plot_euler_characteristic_curve_time(events)

# Skrypt

### (0) Podaj parametr n

In [80]:
n = ...

### (1a) Podaj zbiór punktów

In [ ]:
points=np.array([])

'''forma podania
points=np.array(
    [0,0,0],
    [0.5,1,2],
    [2,3,4],
    [0,1,1])
'''

cech_complex=CechComplex(points)

### (1b) Wygneruj losowy zbiór

#### Podaj paramter k (liczba punktów) oraz rogi prostopadłościany, z którego ma losować

In [68]:
k = ...
low = ...
high = ...

In [76]:
points=generate_points_uniform(n_points=k,low=low,high=high,ambient_dim=n)
cech_complex=CechComplex(points)

### (2) Podaj promień

In [ ]:
r = ...
simplices=cech_complex.build_for_radius(r=r,max_dim=n)

### (3) Wyznaczanie homologii symplicjalnych oraz wyświetlanie ich

In [ ]:
simplical_homology(simplices,n)

#### Wizualizacja


In [79]:
plot_for_dimension(points=points,simplices=simplices,R=r,max_dim=n)

In [ ]:
events=cech_complex.events_from_simplices(simplices)
print_filter(events)

### (5) Wyznaczanie krzywej charakterystyki Euler

In [74]:
plot_euler_characteristic_curve_time(events)